**Author:** Salvador Navas  
**Date:** 2025-06-27

### ESGF

**ESGF (Earth System Grid Federation)** is a globally distributed data infrastructure designed to facilitate access, sharing, and analysis of large-scale climate and Earth system data.

- **Purpose:** To provide access to climate model outputs, reanalysis data, and observational datasets used in climate research, assessments, and impact studies.
- **Developed by:** An international collaboration including institutions such as LLNL (USA), PCMDI, KNMI (Netherlands), BADC (UK), DKRZ (Germany), and others.
- **Key projects hosted:**
  - **CMIP (Coupled Model Intercomparison Project)** data, including CMIP5 and CMIP6 used in IPCC assessments.
  - **CORDEX (Coordinated Regional Downscaling Experiment)** regional climate simulations.
  - **Obs4MIPs**: observational datasets formatted for climate model evaluation.
  - **Reanalysis data** such as ERA-Interim, ERA5, and MERRA.

#### 💡 **Key characteristics:**

1. **Federated architecture:** ESGF consists of multiple data nodes worldwide, ensuring data redundancy, scalability, and local access in different continents.
2. **Data access protocols:** Uses OpenID for authentication and supports HTTP, OPENDAP, GridFTP for efficient data downloads.
3. **Search and discovery:** Users can search datasets by variable, model, experiment, frequency, and spatial/temporal coverage through ESGF portals.
4. **Massive data volume:** Hosts petabytes of climate simulation data required for model intercomparison, downscaling, and impact assessment studies.
5. **Critical for climate research:** Provides the backbone for IPCC Assessment Reports (AR5, AR6) data analyses and projections.

#### 🔗 **Official website:**

- ESGF Portal: [https://esgf.github.io/](https://esgf.github.io/)

#### 🔍 **Where to find all available variables, models, and scenarios:**

You can explore all available datasets, variables, experiments, and models using the ESGF portals:

- **Main ESGF Portal (LLNL node):** [https://esgf-node.llnl.gov/search/cmip6/](https://esgf-node.llnl.gov/search/cmip6/)
- **Other key portals:**
  - **DKRZ (Germany):** [https://esgf-data.dkrz.de/search/cmip6-dkrz/](https://esgf-data.dkrz.de/search/cmip6-dkrz/)
  - **CEDA (UK):** [https://esgf-index1.ceda.ac.uk/search/cmip6-ceda/](https://esgf-index1.ceda.ac.uk/search/cmip6-ceda/)
  - **PCMDI (USA):** [https://pcmdi.llnl.gov/CMIP6/](https://pcmdi.llnl.gov/CMIP6/)

Using these portals, you can filter by:
- **Project** (e.g., CMIP6, CMIP5, CORDEX)
- **Variable** (e.g., `pr`, `tasmax`, `tasmin`)
- **Experiment ID** (e.g., historical, ssp245, ssp585)
- **Model** (e.g., MIROC6, CNRM-CM6-1, ACCESS-CM2)
- **Frequency** (daily, monthly, etc.)

#### 🔬 **Technical reference:**

- Williams, D. N., et al. (2011). The Earth System Grid Federation: Software framework supporting CMIP5 data analysis and dissemination. *AGU Fall Meeting Abstracts.*


In [ ]:
from pyhydra.utils import interactive_map
from pyhydra.data_sources.climate_change import get_dataset_metadata, get_all_urls, get_combination_if_complete, download_file, process_file
from itertools import product
import pandas as pd
from tqdm import tqdm

## 🗺️ Select area of interest

Draw a **rectangle** on the map to define the spatial subset for download.

In [ ]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

In [ ]:
try:
    # === MAIN WORKFLOW ===

    # Define the list of experiments (historical and future climate scenarios)
    experiments = ["historical", "ssp245", "ssp585"]

    # Define the climate variables of interest
    variables = ["pr", "tasmax", "tasmin"]  # precipitation, maximum and minimum temperature

    # Define the list of models to be used (initially populated, then cleared for automatic detection)
    models = [
        "ACCESS-CM2", "BCC-CSM2-MR", "CanESM5", "CMCC-ESM2",
        "CNRM-ESM2-1", "EC-Earth3", "MPI-ESM1-2-HR",
        "MRI-ESM2-0", "NorESM2-MM", "UKESM1-0-LL"
    ]

    # models = []  # uncomment to auto-detect available models from ESGF (slow without credentials)

    # Define the list of simulation variants (e.g., ensemble member IDs)
    variants = ["r1i1p1f1", "r1i1p1f2"]

    # Container to store all records (download links, metadata, etc.)
    all_records = []

    # 🧠 Auto-detect models or variants if the lists are empty
    if not models or not variants:
        print("Searching for available models and variants...")

        # Define base filters for the metadata query
        base_filters = {
            "project": "CMIP6",
            "table_id": "day",
            "experiment_id": experiments,
            "variable_id": variables
        }

        # Fetch metadata for available datasets
        df_base = get_dataset_metadata(base_filters, limit=3000)

        # Auto-populate models if not defined
        if not models:
            models = sorted(df_base["model"].unique().tolist())

        # Auto-populate variants if not defined
        if not variants:
            variants = sorted(df_base["variant"].unique().tolist())

    # Display selected models and variants
    print(f"Models: {models}")
    print(f"Variants: {variants}")

    # Iterate over all combinations of model, variant, and experiment
    for model, variant, experiment in product(models, variants, experiments):
        print(f"Checking: {model} | {variant} | {experiment}")

        combo = get_combination_if_complete(model, experiment, variant, variables)

        if combo:
            print(f"Valid: {model} {variant} {experiment}")
            for row in combo:
                urls_info = get_all_urls(row["dataset_id"])
                for u in urls_info:
                    all_records.append({
                        "model": row["model"],
                        "experiment": row["experiment"],
                        "variant": row["variant"],
                        "variable": row["variable"],
                        "dataset_id": row["dataset_id"],
                        "version": row["version"],
                        "url": u["url"],
                        "url_type": u["url_type"]
                    })
        else:
            print(f"Incomplete: {model} {variant} {experiment}")

    # Convert results to a DataFrame and export to CSV
    df_out = pd.DataFrame(all_records)
    df_out.to_csv("opendap_links_final.csv", index=False)
    print(f"Exported to 'opendap_links_final.csv' with {len(df_out)} links.")

except Exception as e:
    print(f'[ESGF] Query failed: {e}')
    print('[ESGF] ESGF nodes may be unavailable — configure ESGF credentials and retry.')
    df_out = None


Models: ['ACCESS-CM2', 'BCC-CSM2-MR', 'CanESM5', 'CMCC-ESM2', 'CNRM-ESM2-1', 'EC-Earth3', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM', 'UKESM1-0-LL']
Variants: ['r1i1p1f1', 'r1i1p1f2']
Checking: ACCESS-CM2 | r1i1p1f1 | historical
Valid: ACCESS-CM2 r1i1p1f1 historical
Checking: ACCESS-CM2 | r1i1p1f1 | ssp245
Valid: ACCESS-CM2 r1i1p1f1 ssp245
Checking: ACCESS-CM2 | r1i1p1f1 | ssp585
Valid: ACCESS-CM2 r1i1p1f1 ssp585
Checking: ACCESS-CM2 | r1i1p1f2 | historical
Incomplete: ACCESS-CM2 r1i1p1f2 historical
Checking: ACCESS-CM2 | r1i1p1f2 | ssp245
Incomplete: ACCESS-CM2 r1i1p1f2 ssp245
Checking: ACCESS-CM2 | r1i1p1f2 | ssp585
Incomplete: ACCESS-CM2 r1i1p1f2 ssp585
Checking: BCC-CSM2-MR | r1i1p1f1 | historical
Valid: BCC-CSM2-MR r1i1p1f1 historical
Checking: BCC-CSM2-MR | r1i1p1f1 | ssp245
Valid: BCC-CSM2-MR r1i1p1f1 ssp245
Checking: BCC-CSM2-MR | r1i1p1f1 | ssp585
Valid: BCC-CSM2-MR r1i1p1f1 ssp585
Checking: BCC-CSM2-MR | r1i1p1f2 | historical
Incomplete: BCC-CSM2-MR r1i1p1f2 historical
Check

In [ ]:
if df_out is None or len(coordinates_list) < 4:
    print("[ESGF] No links CSV available or no bounding box drawn — download step skipped.")
else:
    import os

    # === USER CONFIGURATION ===
    CSV_INPUT   = "opendap_links_final.csv"
    path_output = "/workspace/data/esgf/day/"

    lon_min = coordinates_list[1]
    lon_max = coordinates_list[3]
    lat_min = coordinates_list[2]
    lat_max = coordinates_list[0]

    if not os.path.exists(CSV_INPUT):
        raise FileNotFoundError(f"CSV file not found: {CSV_INPUT}")

    df = pd.read_csv(CSV_INPUT)
    df['version_date'] = pd.to_datetime(df['version'], format='%Y%m%d', errors='coerce')
    df_latest = df.sort_values("version_date", ascending=False)

    if "url" not in df_latest.columns or "url_type" not in df_latest.columns:
        raise ValueError("CSV must contain 'url' and 'url_type' columns.")

    df_latest = df_latest[df_latest["url_type"] == "HTTPSERVER"]

    slow_nodes = [
        "esgf-data1.llnl.gov",
        "esgf-data04.diasjp.net",
        "aims3.llnl.gov",
        "esgf.ceda.ac.uk",
        "esgf-data2.llnl.gov",
        "esg.lasg.ac.cn"
    ]

    df_urls_filtered = df_latest[~df_latest['url'].apply(lambda u: any(node in u for node in slow_nodes))]
    urls = df_urls_filtered[["url", "url_type"]].dropna().drop_duplicates().values.tolist()
    print(f"Total number of links to process: {len(urls)}")

    for url, url_type in tqdm(urls):
        process_file(url, path_output, lat_min, lat_max, lon_min, lon_max, url_type)

[ESGF] No links CSV available or no bounding box drawn — download step skipped.
